In [1]:
import numpy as np
import torch
import torch.nn as nn
import torchvision.models as models
import torchvision.transforms as transforms
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt
from scipy import signal
import os

# ── Convert EEG epochs to spectrograms ───────────────────
# Spectrograms = images of frequency content over time
# CNN can then classify them like any other image

def epoch_to_spectrogram(epoch, fs=100, nperseg=64):
    """
    Convert 30s EEG epoch → spectrogram image
    epoch: (3000,) array
    Returns: (H, W) spectrogram
    """
    freqs, times, Sxx = signal.spectrogram(
        epoch, fs=fs, nperseg=nperseg
    )

    # Log scale — compresses dynamic range
    Sxx = np.log1p(Sxx)

    # Normalise to 0-1
    Sxx = (Sxx - Sxx.min()) / (Sxx.max() - Sxx.min() + 1e-8)

    return Sxx.astype(np.float32)


# ── Dataset ───────────────────────────────────────────────
class EEGSpectrogramDataset(Dataset):
    def __init__(self, epochs, labels, transform=None):
        self.epochs    = epochs
        self.labels    = labels
        self.transform = transform

    def __len__(self):
        return len(self.epochs)

    def __getitem__(self, idx):
        epoch = self.epochs[idx]
        label = self.labels[idx]

        # Convert to spectrogram
        spec = epoch_to_spectrogram(epoch)

        # Resize to 3-channel image for ResNet
        # ResNet expects RGB — repeat spectrogram 3 times
        spec_3ch = np.stack([spec, spec, spec], axis=0)
        spec_tensor = torch.FloatTensor(spec_3ch)

        if self.transform:
            spec_tensor = self.transform(spec_tensor)

        return spec_tensor, label


# ── Load your Sleep-EDF data ──────────────────────────────
CURRENT_DIR   = os.path.dirname(os.path.abspath(__file__))
PROJECT_ROOT  = os.path.dirname(CURRENT_DIR)
PROCESSED_DIR = os.path.join(PROJECT_ROOT, "data", "processed")

epochs_all = np.load(os.path.join(PROCESSED_DIR,
                                   "epochs_all.npy"))
labels_all = np.load(os.path.join(PROCESSED_DIR,
                                   "labels_all.npy"))

# Visualise a few spectrograms first
stage_names = ['Wake','N1','N2','N3','REM']
fig, axes   = plt.subplots(1, 5, figsize=(15, 4))

for stage in range(5):
    idx  = np.where(labels_all == stage)[0][0]
    spec = epoch_to_spectrogram(epochs_all[idx])
    axes[stage].imshow(spec, aspect='auto',
                       origin='lower', cmap='viridis')
    axes[stage].set_title(stage_names[stage])
    axes[stage].set_xlabel('Time')
    axes[stage].set_ylabel('Frequency (Hz)')

plt.suptitle('EEG spectrograms by sleep stage', fontsize=12)
plt.tight_layout()
plt.show()
# Look at this carefully — you should see visible differences
# between REM (theta dominant) and N3 (delta dominant)


# ── Build CNN classifier for EEG spectrograms ─────────────
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight

X_tr, X_te, y_tr, y_te = train_test_split(
    epochs_all, labels_all,
    test_size=0.2, random_state=42, stratify=labels_all
)

train_ds = EEGSpectrogramDataset(X_tr, y_tr)
test_ds  = EEGSpectrogramDataset(X_te, y_te)

train_loader = DataLoader(train_ds, batch_size=32, shuffle=True)
test_loader  = DataLoader(test_ds,  batch_size=128)

# Transfer learning — ResNet18 for EEG spectrograms
model = models.resnet18(pretrained=True)

# Freeze early layers — they already know edges/textures
for name, param in model.named_parameters():
    if 'layer3' not in name and \
       'layer4' not in name and \
       'fc'     not in name:
        param.requires_grad = False

# New head for 5 sleep stages
model.fc = nn.Sequential(
    nn.Dropout(0.4),
    nn.Linear(512, 128),
    nn.ReLU(),
    nn.Linear(128, 5)
)

# Weighted loss for class imbalance
weights = compute_class_weight(
    'balanced',
    classes=np.unique(y_tr),
    y=y_tr
)
criterion = nn.CrossEntropyLoss(
    weight=torch.FloatTensor(weights)
)

optimizer = torch.optim.Adam([
    {'params': [p for n, p in model.named_parameters()
                if 'fc' not in n and p.requires_grad],
     'lr': 1e-4},
    {'params': model.fc.parameters(), 'lr': 1e-3}
])

# Train
print("Training CNN on EEG spectrograms...\n")
best_acc = 0

for epoch in range(20):
    model.train()
    total_loss, correct, total = 0, 0, 0

    for X_batch, y_batch in train_loader:
        optimizer.zero_grad()
        out  = model(X_batch)
        loss = criterion(out, y_batch)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        correct    += (out.argmax(1) == y_batch).sum().item()
        total      += len(y_batch)

    # Evaluate
    model.eval()
    all_preds = []
    with torch.no_grad():
        for X_batch, y_batch in test_loader:
            preds = model(X_batch).argmax(1)
            all_preds.extend(preds.numpy())

    all_preds = np.array(all_preds)
    test_acc  = (all_preds == y_te).mean()

    if test_acc > best_acc:
        best_acc = test_acc
        torch.save(model.state_dict(),
                   os.path.join(PROJECT_ROOT, 'models',
                                'cnn_eeg_best.pth'))
        flag = " ← best"
    else:
        flag = ""

    print(f"Epoch {epoch+1:2d} | "
          f"Loss: {total_loss/len(train_loader):.4f} | "
          f"Train: {correct/total:.4f} | "
          f"Test: {test_acc:.4f}{flag}")

print(f"\nBest: {best_acc:.4f}")
print("\n=== Classification Report ===")
from sklearn.metrics import classification_report
print(classification_report(y_te, all_preds,
      target_names=stage_names))

NameError: name '__file__' is not defined